In [1]:
import sys
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

import numpy as np
import pickle
import torch
import time

from typing import List
from pathlib import Path
import json

from GEMS_TCO import kernels_vecchia
from GEMS_TCO import debiased_whittle
from GEMS_TCO import alg_optimization, BaseLogger
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data_dynamic_processed, exact_location_filter

Load daily data applying max-min ordering

In [2]:
space: List[str] = ['1', '1']
lat_lon_resolution = [int(s) for s in space]
mm_cond_number: int = 30
years = ['2024']
month_range = [7] 

output_path = input_path = Path(config.mac_estimates_day_path)
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

lat_range_input=[-3, 2]      
lon_range_input=[121, 131] 

df_map, ord_mm, nns_map, monthly_mean = data_load_instance.load_maxmin_ordered_data_bymonthyear(
    lat_lon_resolution=lat_lon_resolution, 
    mm_cond_number=mm_cond_number,
    years_=years, 
    months_=month_range,
    lat_range=lat_range_input,   
    lon_range=lon_range_input,
    is_whittle=False
)

--- Global Monthly Mean for 2024-7: 257.9726 ---
--- Generating NNS Map for Vecchia (C++ Accelerated) ---


In [6]:
daily_aggregated_tensors_dw = [] 
daily_hourly_maps_dw = []      

daily_aggregated_tensors_vecc = [] 
daily_hourly_maps_vecc = []   

for day_index in range(31):
    hour_start_index = day_index * 8
    hour_end_index = (day_index + 1) * 8
    hour_indices = [hour_start_index, hour_end_index]

    # DW용 (고정 격자 좌표 사용)
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        monthly_mean,
        hour_indices, 
        ord_mm=None,
        dtype=torch.float64, 
        keep_ori=False
    )
    daily_aggregated_tensors_dw.append(day_aggregated_tensor)
    daily_hourly_maps_dw.append(day_hourly_map)

    # Vecchia용 (실제 관측 좌표 사용)
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        monthly_mean,
        hour_indices, 
        ord_mm=ord_mm,
        dtype=torch.float64, 
        keep_ori=True
    )
    daily_aggregated_tensors_vecc.append(day_aggregated_tensor)
    daily_hourly_maps_vecc.append(day_hourly_map)

print(f"Loaded {len(daily_hourly_maps_vecc)} days. Tensor shape (day 0): {daily_aggregated_tensors_vecc[0].shape}")

Loaded 31 days. Tensor shape (day 0): torch.Size([145008, 11])


In [7]:
v = 0.5
LAT_COL, LON_COL, VAL_COL, TIME_COL = 0, 1, 2, 3

# ── Vecchia conditioning 설정 ─────────────────────────────────────────
# nns_map에 저장된 이웃 수(mm_cond_number=30)보다 작아야 함
limit_A = 8    # Set A: 현재 시점 공간 이웃 수
limit_B = 8    # Set B: t-1 공간 이웃 수  (실제 크기 = limit_B + 1)
limit_C = 8    # Set C: t-daily_stride 공간 이웃 수 (실제 크기 = limit_C + 1)
# → max_dim: A=8, AB=17, ABC=26
# 정확도 우선 시: limit_A, limit_B, limit_C = 16, 16, 16 (mm_cond_number >= 16 필요)

# ── Heads 설정 ──────────────────────────────────────────────────────────
nheads = 300

# ── 시간 조건화 설정 ───────────────────────────────────────────────────
daily_stride = 8   # daily_stride >= n_time_steps(=8) → Set C 비활성 (t, t-1만)

# ── L-BFGS 설정 ───────────────────────────────────────────────────────
LBFGS_LR = 1.0
LBFGS_MAX_STEPS = 3
LBFGS_HISTORY_SIZE = 100   
LBFGS_MAX_EVAL = 100

In [8]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

days_list = [20]  # 0-based

for day_idx in days_list:
    print(f'\n{"="*40}')
    print(f'--- Day {day_idx+1} (2024-07-{day_idx+1}) ---')
    print(f'{"="*40}')

    daily_hourly_map_vecc = daily_hourly_maps_vecc[day_idx]

    init_sigmasq    = 13.059
    init_range_lat  = 0.2
    init_range_lon  = 0.25
    init_advec_lat  = 0.0218
    init_range_time = 1.5
    init_advec_lon  = -0.1689
    init_nugget     = 0.247

    init_phi2 = 1.0 / init_range_lon
    init_phi1 = init_sigmasq * init_phi2
    init_phi3 = (init_range_lon / init_range_lat) ** 2
    init_phi4 = (init_range_lon / init_range_time) ** 2

    initial_vals = [
        np.log(init_phi1), np.log(init_phi2), np.log(init_phi3),
        np.log(init_phi4), init_advec_lat, init_advec_lon, np.log(init_nugget)
    ]
    params_list = [
        torch.tensor([val], requires_grad=True, dtype=torch.float64, device=DEVICE)
        for val in initial_vals
    ]

    model_instance = kernels_vecchia.fit_vecchia_lbfgs(
        smooth=v,
        input_map=daily_hourly_map_vecc,
        nns_map=nns_map,
        mm_cond_number=mm_cond_number,
        nheads=nheads,
        limit_A=limit_A, limit_B=limit_B, limit_C=limit_C,
        daily_stride=daily_stride,
    )

    optimizer_vecc = model_instance.set_optimizer(
        params_list,
        lr=LBFGS_LR,
        max_iter=LBFGS_MAX_EVAL,
        history_size=LBFGS_HISTORY_SIZE
    )

    print(f"\n--- Vecchia Optimization (Day {day_idx+1}) ---")
    start_time = time.time()

    out, steps_ran = model_instance.fit_vecc_lbfgs(
        params_list,
        optimizer_vecc,
        max_steps=LBFGS_MAX_STEPS,
        grad_tol=1e-7
    )

    print(f"Finished in {time.time() - start_time:.2f}s. Results: {out}")

Using device: cpu

--- Day 21 (2024-07-21) ---

--- Vecchia Optimization (Day 21) ---
🚀 Pre-computing 3-group Vecchia [A=8, AB=17, ABC=26, stored=1]... [Mean Lat: -0.4801] [Set C: False] ✅ Done. (Heads: 2370, Tails A/AB/ABC: 17769/123825/0)
--- Starting Batched L-BFGS Optimization (GPU) ---
--- Step 1/3 / Loss: 1.268006 ---
  Param 0: Value=2.8958, Grad=2.5155533278224977e-06
  Param 1: Value=0.8673, Grad=-5.991357775003597e-07
  Param 2: Value=0.5995, Grad=1.6886027069435046e-07
  Param 3: Value=-2.5103, Grad=2.3654959431130694e-07
  Param 4: Value=0.0128, Grad=1.3626258751689435e-06
  Param 5: Value=-0.1361, Grad=1.658339403690784e-06
  Param 6: Value=0.5750, Grad=3.1306243375274294e-06
  Max Abs Grad: 3.130624e-06
------------------------------
--- Step 2/3 / Loss: 1.243322 ---
  Param 0: Value=2.8958, Grad=2.5155533278224977e-06
  Param 1: Value=0.8673, Grad=-5.991357775003597e-07
  Param 2: Value=0.5995, Grad=1.6886027069435046e-07
  Param 3: Value=-2.5103, Grad=2.3654959431130694

## Previous Run Results (reference)

**irr. (-3,2 121,131, intercept, time dummies, latitude)**

Final Interpretable Params:
- sigma_sq: 14.557, range_lon: 0.363, range_lat: 0.340, range_time: 1.831
- advec_lat: -0.001, advec_lon: -0.166, nugget: 0.250
- Finished in 67.75s

**regular (hide nugget)**

- sigma_sq: 13.882, range_lon: 0.331, range_lat: 0.305, range_time: 1.646
- advec_lat: -0.001, advec_lon: -0.155, nugget: ~0
- Finished in 141.61s

irr ar(1) offset logic 이용했을때 사실 차이 없고 latitude centering 했을 때 소수점 10째자리에서 살짝 개선

fit debiased whittle

In [ ]:
DEVICE_DW = torch.device("cpu")

dwl = debiased_whittle.debiased_whittle_likelihood()
TAPERING_FUNC = dwl.cgn_hamming 
NUM_RUNS = 1
DWL_MAX_STEPS = 15

days_list = [10]  # 0-based

for day_idx in days_list:
    print(f'\n{"="*50}')
    print(f'--- DW: Day {day_idx+1} (2024-07-{day_idx+1}) ---')
    print(f'{"="*50}')

    start_time = time.time()

    try:
        daily_aggregated_tensor_dw = daily_aggregated_tensors_dw[day_idx].to(DEVICE_DW)

        if daily_aggregated_tensor_dw.shape[0] == 0:
            print(f"Skipping Day {day_idx+1}: No data.")
            continue

        init_sigmasq    = 13.059
        init_range_lat  = 0.154 
        init_range_lon  = 0.195
        init_advec_lat  = 0.0218
        init_range_time = 0.7  
        init_advec_lon  = -0.1689
        init_nugget     = 0.247

        raw_init_floats = [init_sigmasq, init_range_lat, init_range_lon, init_range_time,
                           init_advec_lat, init_advec_lon, init_nugget]

        # STEP 1: Pre-process
        db = debiased_whittle.debiased_whittle_preprocess(
            daily_aggregated_tensors_dw, daily_hourly_maps_dw, day_idx=day_idx,
            params_list=raw_init_floats, lat_range=[-3, 2], lon_range=[121.0, 131.0]
        )
        cur_df = db.generate_spatially_filtered_days(-3, 2, 121, 131).to(DEVICE_DW)

        if cur_df.numel() == 0 or cur_df.shape[1] <= max(LAT_COL, LON_COL, VAL_COL, TIME_COL):
            print(f"Error: Data for Day {day_idx+1} is empty or invalid.")
            continue

        unique_times = torch.unique(cur_df[:, TIME_COL])
        time_slices_list = [cur_df[cur_df[:, TIME_COL] == t_val] for t_val in unique_times]

        # STEP 2: Pre-compute J-vector, periodogram, taper autocorr
        print("Pre-computing J-vector (Hamming taper)...")
        J_vec, n1, n2, p_time, taper_grid = dwl.generate_Jvector_tapered(
            time_slices_list, tapering_func=TAPERING_FUNC,
            lat_col=LAT_COL, lon_col=LON_COL, val_col=VAL_COL, device=DEVICE_DW
        )

        if J_vec is None or J_vec.numel() == 0 or n1 == 0 or n2 == 0 or p_time == 0:
            print(f"Error: J-vector generation failed for Day {day_idx+1}.")
            continue

        I_sample = dwl.calculate_sample_periodogram_vectorized(J_vec)
        taper_autocorr_grid = dwl.calculate_taper_autocorrelation_fft(taper_grid, n1, n2, DEVICE_DW)

        if torch.isnan(I_sample).any() or torch.isnan(taper_autocorr_grid).any():
            print("Error: NaN in periodogram or taper autocorrelation.")
            continue

        print(f"Grid: {n1}x{n2}, {p_time} time points.")

        # STEP 3: Optimization
        all_final_results, all_final_losses = [], []

        for i in range(NUM_RUNS):
            print(f"\n{'='*30} Run {i+1}/{NUM_RUNS} {'='*30}")

            init_phi2 = 1.0 / init_range_lon
            init_phi1 = init_sigmasq * init_phi2
            init_phi3 = (init_range_lon / init_range_lat) ** 2
            init_phi4 = (init_range_lon / init_range_time) ** 2

            initial_params_values = [
                np.log(init_phi1), np.log(init_phi2), np.log(init_phi3), np.log(init_phi4),
                init_advec_lat, init_advec_lon, np.log(init_nugget)
            ]

            from torch.nn import Parameter
            params_list = [
                Parameter(torch.tensor([val], dtype=torch.float64, device=DEVICE_DW))
                for val in initial_params_values
            ]

            optimizer = torch.optim.LBFGS(
                params_list, lr=1.0, max_iter=DWL_MAX_STEPS, history_size=100,
                line_search_fn="strong_wolfe", tolerance_grad=1e-5
            )

            nat_params_str, phi_params_str, raw_params_str, loss, steps_run = dwl.run_lbfgs_tapered(
                params_list=params_list, optimizer=optimizer,
                I_sample=I_sample, n1=n1, n2=n2, p_time=p_time,
                taper_autocorr_grid=taper_autocorr_grid,
                max_steps=DWL_MAX_STEPS, device=DEVICE_DW
            )

            if loss is not None:
                all_final_results.append((nat_params_str, phi_params_str, raw_params_str))
                all_final_losses.append(loss)
            else:
                all_final_losses.append(float('inf'))

        # STEP 4: Summary
        valid_losses = [l for l in all_final_losses if l != float('inf')]
        if not valid_losses:
            print(f"All runs failed for Day {day_idx+1}.")
        else:
            best_idx = all_final_losses.index(min(valid_losses))
            best = all_final_results[best_idx]
            print(f"\nBest loss: {min(valid_losses)} (run {best_idx+1}, {steps_run} steps)")
            print(f"Natural scale: {best[0]}")
            print(f"Raw log scale: {best[2]}")

        print(f"\nTotal time: {time.time() - start_time:.2f}s")

    except Exception as e:
        import traceback
        print(f"Day {day_idx+1} Failed: {e}")
        traceback.print_exc()